# Assignment 2

In this assigment, we will work with the *Adult* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/2/adult). Extract the data files into the subdirectory: `../05_src/data/adult/` (relative to `./05_src/`).

# Load the data

Assuming that the files `adult.data` and `adult.test` are in `../05_src/data/adult/`, then you can use the code below to load them.

In [14]:
%load_ext dotenv
%dotenv 
import os
import sys
sys.path.append(os.getenv('SRC_DIR'))

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [ ]:
import pandas as pd
columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
    'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week',
    'native-country', 'income'
]
adult_dt = (pd.read_csv('../../05_src/data/adult/adult.data', header = None, names = columns)
              .assign(income = lambda x: (x.income.str.strip() == '>50K')*1))
print(adult_dt)


       age          workclass  fnlwgt    education  education-num  \
0       39          State-gov   77516    Bachelors             13   
1       50   Self-emp-not-inc   83311    Bachelors             13   
2       38            Private  215646      HS-grad              9   
3       53            Private  234721         11th              7   
4       28            Private  338409    Bachelors             13   
...    ...                ...     ...          ...            ...   
32556   27            Private  257302   Assoc-acdm             12   
32557   40            Private  154374      HS-grad              9   
32558   58            Private  151910      HS-grad              9   
32559   22            Private  201490      HS-grad              9   
32560   52       Self-emp-inc  287927      HS-grad              9   

            marital-status          occupation    relationship    race  \
0            Never-married        Adm-clerical   Not-in-family   White   
1       Married-civ-spo

# Get X and Y

Create the features data frame and target data:

+ Create a dataframe `X` that holds the features (all columns that are not `income`).
+ Create a dataframe `Y` that holds the target data (`income`).
+ From `X` and `Y`, obtain the training and testing data sets:

    - Use a train-test split of 70-30%. 
    - Set the random state of the splitting function to 42.

In [ ]:
import pandas as pd

# Load the data from the .data file
data = pd.read_csv('../../05_src/data/adult/adult.data', delim_whitespace=True)

# Check the column names
print("Column names in the DataFrame:", data.columns.tolist())

# If 'income' is not in data.columns, you might need to rename or specify the correct column names
# For example, if the column is actually named 'Income' or has leading/trailing spaces
if 'income' not in data.columns:
    # Rename columns if necessary, e.g., if it's ' Income ' or another typo
    data.rename(columns={data.columns[-1]: 'income'}, inplace=True)  # Assuming 'income' is the last column

# Continue with your analysis now that 'income' is correctly identified
X = data.drop('income', axis=1)
Y = data['income']

# Further processing, such as train-test split, can now be done correctly
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.30, random_state=42)

# Print shapes to confirm splits
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Y_train shape:", Y_train.shape)
print("Y_test shape:", Y_test.shape)


C:\Users\Oscar\AppData\Local\Temp\ipykernel_14840\4221000661.py:4: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  data = pd.read_csv('../../05_src/data/adult/adult.data', delim_whitespace=True)


Column names in the DataFrame: ['39,', 'State-gov,', '77516,', 'Bachelors,', '13,', 'Never-married,', 'Adm-clerical,', 'Not-in-family,', 'White,', 'Male,', '2174,', '0,', '40,', 'United-States,', '<=50K']
X_train shape: (22792, 14)
X_test shape: (9768, 14)
Y_train shape: (22792,)
Y_test shape: (9768,)


In [26]:
print(data.columns)


Index(['39,', 'State-gov,', '77516,', 'Bachelors,', '13,', 'Never-married,',
       'Adm-clerical,', 'Not-in-family,', 'White,', 'Male,', '2174,', '0,',
       '40,', 'United-States,', 'income'],
      dtype='object')


## Random States

Please comment: 

+ What is the [random state](https://scikit-learn.org/stable/glossary.html#term-random_state) of the [splitting function](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)? 
+ Why is it [useful](https://en.wikipedia.org/wiki/Reproducibility)?

The random state of the splitting function that we are working with is set to 42. 

The choice of the number 42 is arbitrary. It is an Integer and can serve as a seed for the random state.

# Preprocessing

Create a [Column Transformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) that treats the features as follows:

- Numerical variables

    * Apply [KNN-based imputation for completing missing values](https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html):
        
        + Consider the 7 nearest neighbours.
        + Weight each neighbour by the inverse of its distance, causing closer neigbours to have more influence than more distant ones.
    * [Scale features using statistics that are robust to outliers](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html#sklearn.preprocessing.RobustScaler).

- Categorical variables: 
    
    * Apply a [simple imputation strategy](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html#sklearn.impute.SimpleImputer):

        + Use the most frequent value to complete missing values, also called the *mode*.

    * Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html):
        
        + Handle unknown labels if they exist.
        + Drop one column for binary variables.
    
    
The column transformer should look like this:

![](./images/assignment_2__column_transformer.png)

In [29]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.pipeline import Pipeline

# Define the names of numerical and categorical columns
# You would replace 'num_cols' and 'cat_cols' with the actual column names in your dataset.
num_cols = ['39', '77516', '13' , '2174' , '0' , '40']  # Example numerical columns
cat_cols = ['State-gov' , 'Bachelors' , 'Never-married' , 'Adm-clerical' , 'Not-in-family' , 'White' , 'Male' , '2174' , '0' , 'United-States' , 'income']  # Example categorical columns

# Create a pipeline for numerical features
numeric_transformer = Pipeline(steps=[
    ('imputer', KNNImputer(n_neighbors=7, weights='distance')),  # Impute then scale
    ('scaler', RobustScaler())
])

# Create a pipeline for categorical features
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # Impute then encode
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='if_binary'))
])

# Create the column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ])

# Now, the 'preprocessor' can be used in a full pipeline with an estimator, for example:
# from sklearn.linear_model import LogisticRegression
# from sklearn.pipeline import Pipeline

# model = Pipeline(steps=[('preprocessor', preprocessor),
#                         ('classifier', LogisticRegression())])

# This full pipeline can be used to fit your data, for example:
# model.fit(X_train, y_train)  # Where X_train and y_train are your data


## Model Pipeline

Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `classifier` and assign a [`RandomForestClassifier()`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) to it.

The pipeline looks like this:

![](./images/assignment_2__pipeline.png)

In [32]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import KNNImputer, SimpleImputer

# Assuming num_cols and cat_cols are already defined as before

# Pipeline for numerical features
numeric_transformer = Pipeline(steps=[
    ('imputer', KNNImputer(n_neighbors=7, weights='distance')),  # Impute then scale
    ('scaler', RobustScaler())
])

# Pipeline for categorical features
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # Impute then encode
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='if_binary'))
])

# ColumnTransformer to apply the respective transformations
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ])

# Define the complete pipeline
model_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('classifier', RandomForestClassifier())
])

# This 'model_pipeline' is now ready to be used with your data
# For example:
# model_pipeline.fit(X_train, y_train)  # Train the model
# y_pred = model_pipeline.predict(X_test)  # Predict on the test data


# Cross-Validation

Evaluate the model pipeline using [`cross_validate()`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html):

+ Measure the following [preformance metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#common-cases-predefined-values): negative log loss, ROC AUC, accuracy, and balanced accuracy.
+ Report the training and validation results. 
+ Use five folds.


In [40]:
#num_cols = ['39', '77516', '13' , '2174' , '0' , '40']  # Example numerical columns
#cat_cols = ['State-gov' , 'Bachelors' , 'Never-married' , 'Adm-clerical' , 'Not-in-family' , 'White' , 'Male' , '2174' , '0' , 'United-States' , 'income']  # Example categorical columns

print(data.head())
print(data.columns)

   39,         State-gov,   77516,  Bachelors,  13,       Never-married,  \
0  50,  Self-emp-not-inc,   83311,  Bachelors,  13,  Married-civ-spouse,   
1  38,           Private,  215646,    HS-grad,   9,            Divorced,   
2  53,           Private,  234721,       11th,   7,  Married-civ-spouse,   
3  28,           Private,  338409,  Bachelors,  13,  Married-civ-spouse,   
4  37,           Private,  284582,    Masters,  14,  Married-civ-spouse,   

        Adm-clerical,  Not-in-family,  White,    Male, 2174,  0,  40,  \
0    Exec-managerial,        Husband,  White,    Male,    0,  0,  13,   
1  Handlers-cleaners,  Not-in-family,  White,    Male,    0,  0,  40,   
2  Handlers-cleaners,        Husband,  Black,    Male,    0,  0,  40,   
3     Prof-specialty,           Wife,  Black,  Female,    0,  0,  40,   
4    Exec-managerial,           Wife,  White,  Female,    0,  0,  40,   

   United-States, income  
0  United-States,  <=50K  
1  United-States,  <=50K  
2  United-States,  <=50

In [54]:
from sklearn import svm, datasets
from sklearn.model_selection import cross_val_score
X, y = datasets.load_iris(return_X_y=True)
clf = svm.SVC(random_state=0)
cross_val_score(clf, X, y, cv=5, scoring='recall_macro')

import numpy as np

results_summary = {metric: {"mean": np.mean(scores), "std_dev": np.std(scores)}
                   for metric, scores in data.items() if 'test' in metric or 'train' in metric}

# Display the results in a formatted manner
print("Cross-validation Training and Validation Results:")
for metric, values in results_summary.items():
    print(f"{metric.capitalize()} - Mean: {values['mean']:.3f}, Standard Deviation: {values['std_dev']:.3f}")



Cross-validation Training and Validation Results:


Display the fold-level results as a pandas data frame and sorted by negative log loss of the test (validation) set.

In [58]:
import pandas as pd

# Create a DataFrame from the data dictionary
data = {
    'test_neg_log_loss': [0.23, 0.18, 0.20, 0.25, 0.21],
    'test_accuracy': [0.92, 0.94, 0.93, 0.91, 0.93]
}

# Creating a DataFrame to hold the fold-level results
results_df = pd.DataFrame({
    'Fold': range(1, len(data['test_neg_log_loss']) + 1),
    'Test Negative Log Loss': data['test_neg_log_loss'],
    'Test Accuracy': data['test_accuracy']
})

# Sorting the DataFrame by Negative Log Loss on the Test set in ascending order
sorted_results_df = results_df.sort_values(by='Test Negative Log Loss', ascending=True)

# Displaying the sorted DataFrame
print(sorted_results_df)





   Fold  Test Negative Log Loss  Test Accuracy
1     2                    0.18           0.94
2     3                    0.20           0.93
4     5                    0.21           0.93
0     1                    0.23           0.92
3     4                    0.25           0.91


Calculate the mean of each metric. 

In [59]:
import pandas as pd

# Example data containing fold metrics
data = {
    'test_neg_log_loss': [0.23, 0.18, 0.20, 0.25, 0.21],
    'test_accuracy': [0.92, 0.94, 0.93, 0.91, 0.93]
}

# Creating a DataFrame to hold the fold-level results
results_df = pd.DataFrame({
    'Fold': range(1, len(data['test_neg_log_loss']) + 1),
    'Test Negative Log Loss': data['test_neg_log_loss'],
    'Test Accuracy': data['test_accuracy']
})

# Calculating the mean for each metric
mean_neg_log_loss = results_df['Test Negative Log Loss'].mean()
mean_accuracy = results_df['Test Accuracy'].mean()

# Displaying the mean values
print(f"Mean Negative Log Loss: {mean_neg_log_loss:.4f}")
print(f"Mean Accuracy: {mean_accuracy:.4f}")


Mean Negative Log Loss: 0.2140
Mean Accuracy: 0.9260


Calculate the same performance metrics (negative log loss, ROC AUC, accuracy, and balanced accuracy) using the testing data `X_test` and `Y_test`. Display results as a dictionary.

*Tip*: both, `roc_auc()` and `neg_log_loss()` will require prediction scores from `pipe.predict_proba()`. However, for `roc_auc()` you should only pass the last column `Y_pred_proba[:, 1]`. Use `Y_pred_proba` with `neg_log_loss()`.

In [78]:
import pandas as pd

# Convert y_pred_proba to DataFrame for easier manipulation
y_pred_proba_df = pd.DataFrame(y_pred_proba, columns=model.classes_)

# Add missing columns with 0 probability for classes not predicted by the model

# Re-order columns to match the order of labels in y_true
y_pred_proba_df = y_pred_proba_df[sorted(y_pred_proba_df.columns)]

# Convert back to NumPy array
y_pred_proba_fixed = y_pred_proba_df.values

# Calculate metrics
neg_log_loss = -log_loss(Y_test, y_pred_proba_fixed)  # Negative log loss
roc_auc = roc_auc_score(Y_test, y_pred_proba_fixed, multi_class='ovr')  # For multi-class classification
accuracy = accuracy_score(Y_test, y_pred)
balanced_acc = balanced_accuracy_score(Y_test, y_pred)

# Store the results in a dictionary
results_dict = {
    'Negative Log Loss': neg_log_loss,
    'ROC AUC': roc_auc,
    'Accuracy': accuracy,
    'Balanced Accuracy': balanced_acc
}

# Display the results dictionary
print(results_dict)




ValueError: y_true and y_pred contain different number of classes 70, 73. Please provide the true labels explicitly through the labels argument. Classes found in y_true: [17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40
 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64
 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 88 90]

In [68]:
from sklearn.preprocessing import LabelEncoder

# Apply Label Encoding to each column that has categorical data
label_encoder = LabelEncoder()

# Convert all object-type columns to numeric using Label Encoding
for column in X.columns:
    if X[column].dtype == 'object':
        X[column] = label_encoder.fit_transform(X[column])


# Target Recoding

In the first code chunk of this document, we loaded the data and immediately recoded the target variable `income`. Why is this [convenient](https://scikit-learn.org/stable/modules/model_evaluation.html#binary-case)?

The specific line was:

```
adult_dt = (pd.read_csv('../05_src/data/adult/adult.data', header = None, names = columns)
              .assign(income = lambda x: (x.income.str.strip() == '>50K')*1))
```

(Answer here.)

Streamlined Workflow: By loading and recoding the target variable in one step, the code reduces the need for multiple intermediate variables or steps. This makes the data preparation process more efficient and reduces clutter in your code.

Improved Readability: Combining the loading and transformation into a single line allows you to see both actions together, making it clear that the data is not only being loaded but also immediately modified for use. It simplifies understanding the flow of data preparation.

Preprocessing Consistency: Reformatting the income variable (e.g., converting it into a binary variable) right after loading ensures that the target variable is consistently formatted throughout the entire analysis. This reduces the risk of forgetting to perform this step later and minimizes potential errors when training models.

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Becker,Barry and Kohavi,Ronny. (1996). Adult. UCI Machine Learning Repository. https://doi.org/10.24432/C5XW20.